# NC Field NDVI Analysis (Top 10 Fields)

**Date:** 2026-04-18  
**Updated:** Expanded from single field to Top 10 fields by area

## Summary

This notebook analyzes satellite imagery for the top 10 agricultural fields in North Carolina from the field boundaries dataset. The analysis includes:

1. **Field Selection** - Identified top 10 fields by area (excluding the largest which was originally analyzed)
2. **Satellite Data Sources** - Used both Sentinel-2 L2A (10m) and Landsat 7/8/9 C2 L2 (30m)
3. **NDVI Calculation** - Computed Normalized Difference Vegetation Index using Red and Near-Infrared bands
4. **Zonal Statistics** - Extracted per-field NDVI statistics (mean, std, min, max, median, pixel count)
5. **Visualization** - Created color maps for each field's NDVI

**Data Sources:**
- **Sentinel-2 L2A** (10m resolution) - Primary source for 9 fields
- **Landsat 7 C2 L2** (30m resolution) - Used for largest field (osm-260949778)
- **Source:** Microsoft Planetary Computer STAC API
- **Date Range:** April 2025 - April 2026 (most recent 12 months)
- **Cloud Filter:** ≤20% cloud cover

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
from IPython.display import Image, display, Markdown

DATA_DIR = Path("/workspaces/my-farm-advisor/data/workspace")
OUTPUT_DIR = DATA_DIR / "output" / "assignment-05"

## 1. Top 10 Fields Overview

From the NC field boundaries dataset (23 fields), the top 10 fields by area were identified. Note: Field osm-260949778 (largest, 768 acres) was originally analyzed with Landsat 7 in 2023.

In [ ]:
fields = gpd.read_file(DATA_DIR / "NC_field_boundaries_EPSG4326_2026-04-01.geojson")
ndvi_summary = pd.read_csv(OUTPUT_DIR / "ndvi_summary_top10.csv")

merged = fields.merge(ndvi_summary, on='field_id', how='left')
top10 = merged.nlargest(10, 'area_acres')

display_df = top10[['field_id', 'county_name', 'area_acres', 'crop_name', 'ndvi_mean', 'source']].copy()
display_df['area_acres'] = display_df['area_acres'].round(2)
display_df['ndvi_mean'] = display_df['ndvi_mean'].round(4)
print("Top 10 Fields with NDVI Data:")
print(display_df.to_string(index=False))

## 2. NDVI Visualizations

NDVI color maps for all 10 fields using the RdYlGn colormap:
- Red = low NDVI (bare soil/water)
- Yellow = moderate NDVI (sparse vegetation)
- Green = high NDVI (dense vegetation)

In [ ]:
field_viz = [
    ("osm-1153259427", "Union County - 617.77 acres"),
    ("osm-260949778", "Guilford County - 768.05 acres (Landsat 7)"),
    ("osm-813157720", "Person County - 194.99 acres"),
    ("osm-1305439648", "Davidson County - 192.38 acres"),
    ("osm-1386621285", "Rowan County - 187.21 acres"),
    ("osm-1133139440", "Hoke County - 144.39 acres"),
    ("osm-1476971106", "Alamance County - 137.74 acres"),
    ("osm-834363677", "Randolph County - 132.83 acres"),
    ("osm-548794709", "Cumberland County - 99.31 acres"),
    ("osm-199889806", "Moore County - 98.41 acres"),
]

for field_id, desc in field_viz:
    png_path = OUTPUT_DIR / f"{field_id}_NDVI_color.png"
    if png_path.exists():
        display(Markdown(f"### {desc}"))
        display(Image(filename=str(png_path), width=600))

## 3. NDVI Summary Statistics

Complete NDVI statistics for all 10 fields:

In [ ]:
print(ndvi_summary.to_string(index=False))

## 4. NDVI Interpretation

| NDVI Value | Interpretation |
|------------|----------------|
| -1.0 to 0 | Water / non-vegetative |
| 0 to 0.2 | Bare soil / rocks |
| 0.2 to 0.5 | Sparse vegetation |
| 0.5 to 1.0 | Dense green vegetation |

**Key Observations:**
- **Highest NDVI:** Union County (0.40) - indicates dense, healthy vegetation
- **Lowest NDVI:** Hoke County (0.12) - likely bare/stubble fields
- **Seasonal Variation:** Landsat 7 field (Guilford) captured in Nov 2023 shows lower NDVI due to fall conditions

## 5. Zonal Statistics Integration

NDVI data was integrated with soil and weather data through zonal statistics. The script extracts per-field statistics from each NDVI raster and merges with other data sources.

In [ ]:
zonal_df = pd.read_csv(DATA_DIR / "output" / "assignment-07" / "zonal_integration_summary.csv")
zonal_with_ndvi = zonal_df[zonal_df['ndvi_mean'].notna()][['field_id', 'county_name', 'area_acres', 'om_mean', 'ph_mean', 'ndvi_mean', 'gdd_total']]
zonal_with_ndvi = zonal_with_ndvi.round(3)
print("Integrated Zonal Statistics (Fields with NDVI):")
print(zonal_with_ndvi.to_string(index=False))

## 6. Satellite Data Sources Used

This table documents the satellite data sources used for each field:

| Field ID | County | Source | Collection | Resolution | Date |
|----------|--------|--------|-------------|------------|------|
| osm-260949778 | Guilford | Landsat 7 | landsat-c2-l2 | 30m | Nov 4, 2023 |
| osm-1153259427 | Union | Sentinel-2 | sentinel-2-l2a | 10m | Apr 2025 - Apr 2026 |
| osm-813157720 | Person | Sentinel-2 | sentinel-2-l2a | 10m | Apr 2025 - Apr 2026 |
| osm-1305439648 | Davidson | Sentinel-2 | sentinel-2-l2a | 10m | Apr 2025 - Apr 2026 |
| osm-1386621285 | Rowan | Sentinel-2 | sentinel-2-l2a | 10m | Apr 2025 - Apr 2026 |
| osm-1133139440 | Hoke | Sentinel-2 | sentinel-2-l2a | 10m | Apr 2025 - Apr 2026 |
| osm-1476971106 | Alamance | Sentinel-2 | sentinel-2-l2a | 10m | Apr 2025 - Apr 2026 |
| osm-834363677 | Randolph | Sentinel-2 | sentinel-2-l2a | 10m | Apr 2025 - Apr 2026 |
| osm-548794709 | Cumberland | Sentinel-2 | sentinel-2-l2a | 10m | Apr 2025 - Apr 2026 |
| osm-199889806 | Moore | Sentinel-2 | sentinel-2-l2a | 10m | Apr 2025 - Apr 2026 |

**Processing Details:**
- Sentinel-2: Used B04 (Red) and B08 (NIR) bands, applied SCL cloud masking
- Landsat 7: Used B4 (Red) and B5 (NIR) bands, surface reflectance scaling applied
- Both: NDVI = (NIR - Red) / (NIR + Red), clipped to field boundaries

## 7. Scripts Used

The following Python scripts were used to generate the outputs:

| Script | Description |
|--------|-------------|
| `generate_bands.py` | Downloads Landsat bands from Planetary Computer STAC API and clips to field boundary (original largest field) |
| `generate_ndvi.py` | Calculates NDVI from Red and NIR bands, creates NDVI visualization |
| `process_ndvi_batch1.py` | Downloads Sentinel-2 imagery and calculates NDVI for top 10 fields |
| `generate_zonal_stats.py` | Extracts zonal statistics from NDVI rasters and integrates with soil/weather data |

All scripts are saved in `data/workspace/scripts/`.

**Output Files:**
- NDVI rasters: `output/assignment-05/{field_id}_NDVI.tif`
- NDVI visualizations: `output/assignment-05/{field_id}_NDVI_color.png`
- Zonal integration: `output/assignment-07/zonal_integration_summary.csv`
- Combined GeoJSON: `output/assignment-05/NC_field_boundaries_with_ndvi.geojson`